In [8]:
import os
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

giga_key = os.getenv("GIGA_KEY")
if not giga_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

In [9]:
llm = GigaChat(
    credentials=giga_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    scope="GIGACHAT_API_PERS",
    temperature=0.2,
    max_tokens=1000
)

In [10]:
response = llm.invoke("Привет! Как дела?")
print(response.content)

Привет! Всё отлично, готов общаться и помогать. А у тебя как настроение сегодня?


In [11]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)

chain = basic_prompt | llm | StrOutputParser()

In [12]:
test_texts = [
    "Ищу квартиру для семьи из четырех человек на длительный срок",
    "Нужна студия для проживания одного человека рядом с метро",
    "Требуется двухкомнатная квартира для молодой пары",
    "Снимем жилье для троих студентов на учебный год",
    "Семья с двумя детьми ищет просторную квартиру",
    "Сниму недорогое жилье на 6 чел 3 взр и 3 дет 30.07-09.08",
    "Здравствуйте. Снимем жилье с 12.07 по 17.07. 2 взрослых и 1 ребенок 6 лет, со всеми удобствами.",
    "Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.",
    "Ищем жильё на 4 взрослых с 31.08 по 07.09, желательно центр",
    "Срочно ищем жилье с 13.06 по 20.06 нас 3 человека(все взрослые). Желательно с удобствами в номере.",
    "Ищем жилье для двоих, 2-местный номер с удобствами, с 7 сентября не менее чем на неделю",
    "Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июля по 21 июля",
    "Ищем жильё 4 чел. 2 взрослых и 2 детей, сумма не более 65000 р. Квартира 2-х ком.",
    "Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от пляжа",
    "Сниму жильё на 2 недели. Конец августа. Два взрослых человека. По адекватной цене."
]

for text in test_texts:
    result = chain.invoke({"text": text})
    print(f"Текст: {text}")
    print(f"Результат: {result}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек на длительный срок
Результат: 4
---
Текст: Нужна студия для проживания одного человека рядом с метро
Результат: 1
---
Текст: Требуется двухкомнатная квартира для молодой пары
Результат: 2
---
Текст: Снимем жилье для троих студентов на учебный год
Результат: 3
---
Текст: Семья с двумя детьми ищет просторную квартиру
Результат: 4
---
Текст: Сниму недорогое жилье на 6 чел 3 взр и 3 дет 30.07-09.08
Результат: 6
---
Текст: Здравствуйте. Снимем жилье с 12.07 по 17.07. 2 взрослых и 1 ребенок 6 лет, со всеми удобствами.
Результат: 3
---
Текст: Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.
Результат: 2
---
Текст: Ищем жильё на 4 взрослых с 31.08 по 07.09, желательно центр
Результат: 4
---
Текст: Срочно ищем жилье с 13.06 по 20.06 нас 3 человека(все взрослые). Желательно с удобствами в номере.
Результат: 3
---
Текст: Ищем жилье для двоих, 2-местный номер с удобствами, с 7 сентября не менее чем на неделю
Результат: 2
---
Текст: Ище

In [13]:
import pandas as pd

df = pd.read_csv('rental_29.csv', sep=';')
df.head()

,text,amount
0,Ищу квартиру для семьи из четырех человек на д...,4
1,Нужна студия для проживания одного человека ря...,1
2,Требуется двухкомнатная квартира для молодой пары,2
3,Снимем жилье для троих студентов на учебный год,3
4,Семья с двумя детьми ищет просторную квартиру,4


In [14]:
results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(result.strip())
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_29_with_results.csv", index=False, encoding="utf-8-sig")
print("Готово!")
df

Готово!


,text,amount,result
0,Ищу квартиру для семьи из четырех человек на д...,4,4
1,Нужна студия для проживания одного человека ря...,1,1
2,Требуется двухкомнатная квартира для молодой пары,2,2
3,Снимем жилье для троих студентов на учебный год,3,3
4,Семья с двумя детьми ищет просторную квартиру,4,4
5,Однокомнатная квартира для одного командировоч...,1,1
6,Ищем жилье для пятерых коллег на время проекта,5,5
7,Молодая семья с новорожденным ребенком ищет кв...,3,4
8,Нужна квартира для троих — я и двое друзей,3,3
9,"Семья из шести человек переезжает, нужен больш...",6,6


In [15]:
df["result_num"] = pd.to_numeric(df["result"], errors="coerce")
df[["amount", "result", "result_num"]].head(15)

,amount,result,result_num
0,4,4,4
1,1,1,1
2,2,2,2
3,3,3,3
4,4,4,4
5,1,1,1
6,5,5,5
7,3,4,4
8,3,3,3
9,6,6,6


In [16]:
correct = (df["amount"] == df["result_num"]).sum()
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

Верных ответов: 14
Ошибок: 1
Точность: 93.3%


In [17]:
mistakes = df[df["amount"] != df["result_num"]][["amount", "text", "result"]]
mistakes

,amount,text,result
7,3,Молодая семья с новорожденным ребенком ищет кв...,4


In [18]:
from langchain_core.output_parsers import JsonOutputParser
import json

json_schema = {
    "type": "object",
    "properties": {
        "count_adults": {"type": "integer", "description": "Количество взрослых"},
        "count_children": {"type": "integer", "description": "Количество детей"},
        "start_date": {"type": "string", "description": "Дата заезда или null"},
        "nights": {"type": "integer", "description": "Количество ночей или null"},
        "price_per_day": {"type": "number", "description": "Цена в сутки или null"},
        "remarks": {"type": "string", "description": "Особые пожелания или null"}
    },
    "required": ["count_adults", "count_children"]
}

json_parser = JsonOutputParser(schema=json_schema)

In [19]:
json_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Ты — эксперт по анализу заявок на аренду жилья.
Извлеки информацию из заявки используя примеры ниже.

ПРИМЕРЫ:
Заявка: "Семья из 2 взрослых и 1 ребенка ищет квартиру с 01.07 на 14 ночей, бюджет 2000 в сутки, нужна парковка"
Ответ: {{"count_adults": 2, "count_children": 1, "start_date": "01.07", "nights": 14, "price_per_day": 2000, "remarks": "парковка"}}

Заявка: "Молодая пара снимет студию с 15 августа на неделю"
Ответ: {{"count_adults": 2, "count_children": 0, "start_date": "15.08", "nights": 7, "price_per_day": null, "remarks": null}}

Заявка: "Один человек ищет комнату рядом с метро"
Ответ: {{"count_adults": 1, "count_children": 0, "start_date": null, "nights": null, "price_per_day": null, "remarks": "рядом с метро"}}

Теперь проанализируй эту заявку и верни ТОЛЬКО JSON без пояснений:
Заявка: {text}
Ответ:""",
)

json_chain = json_prompt | llm | json_parser

In [20]:
test_json = [
    "Семья с двумя детьми ищет квартиру с 1 июня на 2 недели, бюджет 5000 в сутки",
    "Молодая пара снимет студию рядом с метро, нужна парковка",
    "Трое студентов ищут трёхкомнатную квартиру на учебный год с сентября",
    "Один командировочный ищет жилье с 10.07 по 17.07, до 1500 в сутки",
    "Семья 2 взрослых и 3 детей, с 25 июля на месяц, нужна детская кроватка"
]

for text in test_json:
    try:
        result = json_chain.invoke({"text": text})
        print(f"Заявка: {text}")
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print("---")
    except Exception as e:
        print(f"Ошибка: {e}")

Заявка: Семья с двумя детьми ищет квартиру с 1 июня на 2 недели, бюджет 5000 в сутки
{
  "count_adults": 2,
  "count_children": 2,
  "start_date": "01.06",
  "nights": 14,
  "price_per_day": 5000,
  "remarks": null
}
---
Заявка: Молодая пара снимет студию рядом с метро, нужна парковка
{
  "count_adults": 2,
  "count_children": 0,
  "start_date": null,
  "nights": null,
  "price_per_day": null,
  "remarks": "рядом с метро, парковка"
}
---
Заявка: Трое студентов ищут трёхкомнатную квартиру на учебный год с сентября
{
  "count_adults": 3,
  "count_children": 0,
  "start_date": "сентябрь",
  "nights": 200,
  "price_per_day": null,
  "remarks": "трёхкомнатная квартира"
}
---
Заявка: Один командировочный ищет жилье с 10.07 по 17.07, до 1500 в сутки
{
  "count_adults": 1,
  "count_children": 0,
  "start_date": "10.07",
  "nights": 8,
  "price_per_day": 1500,
  "remarks": null
}
---
Заявка: Семья 2 взрослых и 3 детей, с 25 июля на месяц, нужна детская кроватка
{
  "count_adults": 2,
  "count_c

In [21]:
json_results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = json_chain.invoke({"text": text})
        json_results.append(result)
    except Exception as e:
        json_results.append({
            "count_adults": None,
            "count_children": None,
            "start_date": None,
            "nights": None,
            "price_per_day": None,
            "remarks": None
        })

df_json = pd.DataFrame(json_results)
df_json["text"] = df["text"]
df_json["amount_true"] = df["amount"]
df_json["total_people"] = df_json["count_adults"].fillna(0) + df_json["count_children"].fillna(0)

df_json.to_csv("rental_29_json_results.csv", index=False, encoding="utf-8-sig")
print("Готово!")
df_json

Готово!


,count_adults,count_children,start_date,nights,price_per_day,remarks,text,amount_true,total_people
0,4,0,NaN,NaN,None,NaN,Ищу квартиру для семьи из четырех человек на д...,4,4
1,1,0,NaN,NaN,None,рядом с метро,Нужна студия для проживания одного человека ря...,1,1
2,2,0,NaN,NaN,None,двухкомнатная квартира,Требуется двухкомнатная квартира для молодой пары,2,2
3,3,0,01.09,200.0,None,NaN,Снимем жилье для троих студентов на учебный год,3,3
4,2,2,NaN,NaN,None,просторная квартира,Семья с двумя детьми ищет просторную квартиру,4,4
5,1,0,NaN,NaN,None,однокомнатная квартира,Однокомнатная квартира для одного командировоч...,1,1
6,5,0,NaN,NaN,None,для пятерых коллег на время проекта,Ищем жилье для пятерых коллег на время проекта,5,5
7,2,1,NaN,NaN,None,с новорожденным ребенком,Молодая семья с новорожденным ребенком ищет кв...,3,3
8,3,0,NaN,NaN,None,NaN,Нужна квартира для троих — я и двое друзей,3,3
9,6,0,NaN,NaN,None,нужен большой дом,"Семья из шести человек переезжает, нужен больш...",6,6


In [22]:
total = len(df_json)
correct_total = (df_json["total_people"] == df["amount"]).sum()
accuracy_total = correct_total / total

print(f"Точность по общему количеству: {accuracy_total:.1%}")
print(f"Верных: {correct_total} из {total}")

Точность по общему количеству: 100.0%
Верных: 15 из 15
